<a href="https://colab.research.google.com/github/GiovanniPasq/agentic-rag-for-dummies/blob/main/notebooks/observability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG 系统可观测性

本指南将介绍：在 agentic RAG 流水线中**为什么可观测性很重要**，对比 2025-2026 年主流平台，并演示如何在本项目中使用 **Langfuse** 进行落地集成。

## 1. 为什么可观测性很重要

传统应用的失败通常“显式可见”：HTTP 500、堆栈报错、超时。基于 LLM 的系统往往是**静默失败**：Agent 可能选错工具、检索到过期文档、幻觉出错误关联，并且还会自信地给出结果，却不抛出任何异常。

面向 agentic 系统的可观测性，能够回答标准 APM 工具无法回答的问题：

- **检索器是否返回了正确分块？**（Context Precision / Recall）
- **生成答案是否基于已检索数据？**（Faithfulness）
- **Agent 调用了哪个工具，调用是否正确？**（Tool Call tracing）
- **图执行中延迟累积在何处？**（Span-level timing）
- **每次查询成本是多少？**（Token-level cost tracking）

## 2. RAG 的关键评估指标

在选择可观测性平台前，先明确你要评估什么。

| 维度 | 指标 | 衡量内容 | 常见失败原因 |
|-----------|--------|------------------|-----------------------|
| **Retrieval** | [Context Precision](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_precision/) | 相关分块是否比无关分块排序更高 | 缺少或错误配置 re-ranker；相关文档虽被检索但排名过低 |
| **Retrieval** | [Context Recall](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/) | 参考答案中的信息是否都被检索上下文覆盖 | 分块过小（过度切分）、领域 embedding 能力弱、检索语义缺口 |
| **Retrieval** | [Context Relevance](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/nvidia_metrics/#context-relevance) | 检索上下文是否与用户问题相关 | embedding 模型不匹配、查询理解差、缺少元数据过滤 |
| **Generation** | [Response Groundedness](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/nvidia_metrics/#response-groundedness) | 响应中每个结论是否都被检索上下文支持 | 系统提示词过宽松、模型倾向于超出上下文进行编造 |
| **Generation** | [Answer Relevance](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/response_relevancy/) | 生成回答是否真正回应用户问题 | 查询误解、缺少查询改写步骤、提示模板过于模糊 |
| **Generation** | [Answer Accuracy](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/nvidia_metrics/#answer-accuracy) | 模型回答与参考真值的一致程度 | 幻觉、检索缺失上下文、知识库过期 |

大多数平台通过 **LLM-as-a-judge**（用更强模型评估输出）计算这些指标。[Ragas](https://docs.ragas.io/) 和 [DeepEval](https://docs.confident-ai.com/) 提供了标准化实现。

## 3. 遥测结构：Session、Trace 与 Span

Agentic 系统会产生**分层且非线性**的执行树。标准遥测模型通常分为三层：

- **Session** - 完整多轮会话。用于跟踪长期记忆一致性与行为漂移。
- **Trace** - Session 内一次完整请求-响应周期，包含耗时和成本数据。
- **Span** - Trace 中的最小工作单元：一次 LLM 调用、一次工具调用、一次检索步骤。

在本项目这类 agentic 图中，一个用户问题可能触发几十个 span：查询改写、并行子 Agent 调用、多次 search/retrieve、上下文压缩和最终聚合。

## 4. 平台对比

| Platform | 类型 | 开源 | 最适用场景 | 价格（免费层） |
|----------|------|------------|----------|---------------------|
| **[Langfuse](https://langfuse.com)** | Tracing & prompt management | 是（核心 MIT；企业版有 EE 功能） | 自托管部署、数据主权、框架无关集成 | Cloud：50K units/月；自托管：无限制 |
| **[LangSmith](https://smith.langchain.com)** | Tracing & debugging | 否（专有） | LangChain/LangGraph 原生工作流、图状态可视化 | 5K traces/月 |
| **[Arize Phoenix](https://phoenix.arize.com)** | MLOps 可观测性 | 是（ELv2，不能作为第三方托管服务二次售卖） | embedding 漂移检测、向量空间分析 | 自托管无限制 |
| **[AgentOps](https://agentops.ai)** | Agent 专项监控 | 是（MIT） | 多 Agent 协作分析、按角色成本追踪 | 提供免费层 |
| **[Braintrust](https://braintrust.dev)** | 端到端评估 + CI/CD | 部分开源（SDK 开源） | 生产到评估的反馈闭环、发布质量门禁 | 提供免费层 |
| **[Helicone](https://helicone.ai)** | AI Gateway / Proxy | 是（Apache 2.0） | 成本优化、语义缓存、多提供商路由 | 10K 请求/月 |

### 集成方式

这些平台主要有三种埋点策略：

1. **SDK-based**（Braintrust、LangSmith）- 在代码中使用装饰器/包装器。内部可见性最深，但耦合更紧。
2. **OpenTelemetry**（Arize Phoenix、Langfuse）- 使用标准 OTEL spans。可与 Datadog/Grafana/Splunk 互通，更具长期兼容性。
3. **Proxy/Gateway**（Helicone）- 网络层拦截。无需改代码，但通常只能看到 LLM 输入输出，难以覆盖 Agent 内部逻辑。

### 应该选哪个？

- **LangChain/LangGraph 技术栈** → LangSmith 原生集成最深、配置最少；Langfuse 是最强开源替代。
- **自托管 / 数据主权** → Langfuse（MIT 核心，自托管无限制）或 Arize Phoenix（ELv2，自托管无限制）。
- **多 Agent 调试** → AgentOps 更专注 Agent 协作模式。
- **大规模成本优化** → Helicone 作为网关，配合语义缓存。
- **CI/CD 质量门禁** → Braintrust 适合自动化回归防护。


## 5. 实践集成：在本项目中使用 Langfuse

本项目采用 Langfuse，因为它是开源、框架无关，并且可以通过回调系统非常轻松地与 LangGraph 集成。

### 工作原理

LangGraph 会把顶层 `graph.stream()` 配置中的 `callbacks` 自动传递到每个节点、子图和 LLM 调用。因此我们只需要在**一个位置**注入 Langfuse 回调处理器（`get_config()` 方法），所有操作都会被追踪。

```
graph.stream(input, config={"callbacks": [langfuse_handler]})
    └─ summarize_history   → llm.invoke() ← traced
    └─ rewrite_query       → llm.invoke() ← traced
    └─ agent (subgraph, via Send())
    │   └─ orchestrator    → llm.invoke() ← traced
    │   └─ tools           → search/retrieve ← traced
    │   └─ compress_context → llm.invoke() ← traced
    └─ aggregate_answers   → llm.invoke() ← traced
```


### 配置步骤

**步骤 1** - 安装依赖：

```bash
pip install langfuse
```

> 如果你是通过 `pip install -r requirements.txt` 安装项目依赖，则已包含 langfuse。

**步骤 2** - 配置环境变量：

```
LANGFUSE_ENABLED=true
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com  # 或你的自托管地址
```

将 `.env.example` 复制为 `.env` 并填写上述变量。

**步骤 3** - 正常运行应用：

```bash
python project/app.py
```

随后可在 Langfuse dashboard 中看到追踪数据。

如需**关闭**追踪，请在 `.env` 中设置 `LANGFUSE_ENABLED=false`。无论是否开启追踪，应用行为一致。


### 集成代码

完整实现分布在 3 个文件中，各自职责如下：

In [ ]:
# core/observability.py — Observability class (simplified view)

from langfuse import get_client
from langfuse.langchain import CallbackHandler

class Observability:
    def __init__(self):
        # Reads LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL
        # from environment variables automatically.
        self._client = get_client()
        self._handler = CallbackHandler()

    def get_handler(self):
        return self._handler

    def flush(self):
        self._client.flush()

# The actual class includes defensive imports (try/except ImportError)
# and graceful fallback when langfuse is not installed or configured.


In [ ]:
# core/rag_system.py

from ingestion.chunking import DocumentChunker
from observability.langfuse import Observability

class RAGSystem:
    def __init__(self):
        self.vector_db = VectorDbManager()
        self.parent_store = ParentStoreManager()
        self.chunker = DocumentChunker()
        self.observability = Observability()  
        # ...

    def get_config(self):
        cfg = {
            "configurable": {"thread_id": self.thread_id},
            "recursion_limit": self.recursion_limit,
        }
        handler = self.observability.get_handler()
        if handler:
            cfg["callbacks"] = [handler]
        return cfg

# LangGraph propagates callbacks to ALL nodes and subgraphs automatically.

In [ ]:
# core/chat_interface.py — Flush after session is cleared

def chat(self, message, history):
    # Simplified view. The actual implementation also handles the human-in-the-loop
    # case (current_state.next) for query clarification interrupts.
    for chunk, metadata in self.rag_system.agent_graph.stream(
        {"messages": [HumanMessage(content=message.strip())]},
        config=self.rag_system.get_config(),
        stream_mode="messages"
    ):
        yield chunk

def clear_session(self):
    self.rag_system.reset_thread()
    self.rag_system.observability.flush()  # Ensures pending traces are sent


### 会追踪哪些内容

按此配置，Langfuse 会捕获：

- 图中所有节点的每一次 LLM 调用（orchestrator、查询改写、摘要、压缩、聚合）
- 结构化输出解析（例如改写步骤中的 `QueryAnalysis`）
- 工具调用（`search_child_chunks`、`retrieve_parent_chunks`）及其参数和结果
- 包括并行子 Agent fan-out 在内的完整图执行流程
- 每个 span 的 token 统计与延迟

## 6. Langfuse 部署方式

### 方案 A：Langfuse Cloud

在 [cloud.langfuse.com](https://cloud.langfuse.com) 注册，创建组织和项目，获取 API Key 并写入 `.env`。免费额度为每月 50K units（units = traces + observations + scores）。数据托管在 Langfuse 服务器（EU/US）。

### 方案 B：使用 Docker Compose 自托管

若需完全数据主权，可克隆并运行官方 Langfuse 栈：

```bash
git clone https://github.com/langfuse/langfuse.git
cd langfuse
docker compose up
```

然后访问 `http://localhost:3000`，创建账号和项目，并在 `.env` 中设置：

```
LANGFUSE_ENABLED=true
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=http://localhost:3000
```

高级配置、Kubernetes 部署和生产加固请参考官方[自托管文档](https://langfuse.com/self-hosting)。


## 7. 延伸阅读

- [Langfuse documentation](https://langfuse.com/docs)
- [LangGraph streaming](https://langchain-ai.github.io/langgraph/how-tos/streaming/)
- [Ragas evaluation framework](https://docs.ragas.io/)
- [DeepEval — LLM testing](https://docs.confident-ai.com/)
- [OpenTelemetry semantic conventions for GenAI](https://opentelemetry.io/docs/specs/semconv/gen-ai/)
